# Part 1: class-conditional GMM reason prediction

This experiment fits a generative model directly for the five independent reason labels. For each reason, we fit one Gaussian mixture to the originally labeled positive trials and another to the originally labeled negative trials. We predict using the two class likelihoods and their training priors.

There is no unlabeled-trial pseudo-labeling step and no logistic-regression refit in this notebook. That makes this a direct generative-classifier experiment. By itself, it does not perform the assignment's required label augmentation.

We first standardize the 150 embeddings and transform them with RBF Kernel PCA. The transform is fit on the training embeddings (including unlabeled rows), but each class-conditional GMM is fit only on the 157 originally labeled rows. Validation labels are used only to evaluate the resulting predictions. Trial IDs are not model features; test labels are never accessed.

Previously saved validation references, using the same nonempty-prediction rule, were 0.8782 sample-wise F1 for the observed-label logistic baseline and 0.8880 for the selected K-means pipeline. Those are context, not runs of this notebook.

In [10]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.special import expit
from sklearn.decomposition import KernelPCA
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

## Find and load the Part 1 files

In [11]:
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
candidates += [folder / "applied-ml-midterm" for folder in candidates.copy()]
candidates.append(Path("/Users/prith/Documents/Cornell/Applied Machine Learning/midterm/applied-ml-midterm"))

PROJECT_DIR = next((folder for folder in candidates if (folder / "part1-data").is_dir()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError(f"Could not find part1-data. Kernel working directory: {cwd}")

ATTEMPT_DIR = PROJECT_DIR / "prith_attempts" / "part_1"
DATA_DIR = PROJECT_DIR / "part1-data"

In [12]:
part1_train = pd.read_csv(DATA_DIR / "part1_train.csv")
part1_val = pd.read_csv(DATA_DIR / "part1_val.csv")
part1_test = pd.read_csv(DATA_DIR / "part1_test.csv")

print("Training:", part1_train.shape)
print("Validation:", part1_val.shape)
print("Test:", part1_test.shape)

Training: (1171, 166)
Validation: (78, 166)
Test: (312, 161)


## Configure the class-conditional mixtures

Each reason gets its own positive and negative density model. The number of components is capped by the number of observed examples in that class. Diagonal covariance limits the number of parameters for these small labeled subsets.

In [13]:
REASONS = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
EMBEDDINGS = [f"emb_{i}" for i in range(150)]

KERNEL_PCA_COMPONENTS = 50
KERNEL_PCA_GAMMA = 1 / 75
KERNEL_PCA_SEED = 42

MAX_COMPONENTS_PER_CLASS = 2
COVARIANCE_TYPE = "diag"
N_INIT = 5
MAX_ITER = 300
REG_COVAR = 1e-4
RANDOM_STATE = 42
REASON_THRESHOLD = 0.50

## Scale and transform the embeddings

Fit each transform on training embeddings only, then apply it to validation and test. The GMM classifier below learns class-specific densities only from training rows whose five labels are all observed.

In [14]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(part1_train[EMBEDDINGS])
X_val_scaled = scaler.transform(part1_val[EMBEDDINGS])
X_test_scaled = scaler.transform(part1_test[EMBEDDINGS])

kernel_pca = KernelPCA(
    n_components=KERNEL_PCA_COMPONENTS,
    kernel="rbf",
    gamma=KERNEL_PCA_GAMMA,
    eigen_solver="arpack",
    random_state=KERNEL_PCA_SEED,
)
X_train = kernel_pca.fit_transform(X_train_scaled)
X_val = kernel_pca.transform(X_val_scaled)
X_test = kernel_pca.transform(X_test_scaled)

print("Generative-model feature shapes:", X_train.shape, X_val.shape, X_test.shape)

Generative-model feature shapes: (1171, 50) (78, 50) (312, 50)


## Select the originally labeled training examples

A training row is usable for the class-conditional likelihoods only when all five reasons are observed as 0 or 1. Rows with `-100` do not contribute to either density model.

In [15]:
known_mask = part1_train[REASONS].isin([0, 1]).all(axis=1).to_numpy()
X_labeled = X_train[known_mask]
y_labeled = part1_train.loc[known_mask, REASONS].to_numpy(dtype=int)

print("Originally labeled rows:", len(X_labeled))
print("Originally unlabeled rows:", int((~known_mask).sum()))
display(pd.DataFrame({
    "negative_count": (y_labeled == 0).sum(axis=0),
    "positive_count": (y_labeled == 1).sum(axis=0),
}, index=REASONS))

Originally labeled rows: 157
Originally unlabeled rows: 1014


,negative_count,positive_count
Safety,141,16
Funding,149,8
Efficacy,140,17
Recruitment,56,101
Other,132,25


## Fit one positive and one negative GMM per reason

For reason `r`, the classifier estimates

`P(r=1 | x) = p(x | r=1) P(r=1) / [p(x | r=1) P(r=1) + p(x | r=0) P(r=0)]`.

The two densities are Gaussian mixtures fit to that reason's observed positives and negatives. Priors are the observed training frequencies. Reasons are modeled independently, so a trial can receive several positive predictions.

In [16]:
def fit_class_conditional_gmms(X, y, reasons, max_components_per_class=2,
                               covariance_type="diag", n_init=5, max_iter=300,
                               reg_covar=1e-4, random_state=42):
    """Fit positive/negative density GMMs for each reason, using observed labels only."""
    X = np.asarray(X)
    y = np.asarray(y)
    if X.ndim != 2 or y.shape != (len(X), len(reasons)):
        raise ValueError("X and y must have matching rows and one column per reason.")
    if not np.isin(y, [0, 1]).all():
        raise ValueError("Only originally observed binary labels may fit class densities.")
    y = y.astype(int)

    fitted = {}
    for column, reason in enumerate(reasons):
        y_reason = y[:, column]
        positive_count = int(y_reason.sum())
        negative_count = int(len(y_reason) - positive_count)
        if min(positive_count, negative_count) < 2:
            raise ValueError(f"{reason} needs at least two observed examples of each class for GMM fitting.")

        class_models = {}
        class_priors = {1: positive_count / len(y_reason), 0: negative_count / len(y_reason)}
        for value, count in [(1, positive_count), (0, negative_count)]:
            X_class = X[y_reason == value]
            n_components = min(max_components_per_class, count)
            mixture = GaussianMixture(
                n_components=n_components,
                covariance_type=covariance_type,
                n_init=n_init,
                max_iter=max_iter,
                reg_covar=reg_covar,
                random_state=random_state,
            )
            mixture.fit(X_class)
            class_models[value] = {
                "model": mixture,
                "training_count": count,
                "n_components": n_components,
                "converged": bool(mixture.converged_),
                "iterations": int(mixture.n_iter_),
            }

        fitted[reason] = {
            "class_models": class_models,
            "class_priors": class_priors,
        }
    return fitted


In [17]:
def predict_reason_probabilities(fitted_models, X, reasons):
    """Return five Bayes posterior probabilities, one independent column per reason."""
    X = np.asarray(X)
    probabilities = np.zeros((len(X), len(reasons)), dtype=float)
    for column, reason in enumerate(reasons):
        entry = fitted_models[reason]
        positive = entry["class_models"][1]["model"]
        negative = entry["class_models"][0]["model"]
        prior_positive = entry["class_priors"][1]
        prior_negative = entry["class_priors"][0]

        log_positive = positive.score_samples(X) + np.log(prior_positive)
        log_negative = negative.score_samples(X) + np.log(prior_negative)
        # expit is a numerically stable sigmoid of the log-joint difference.
        probabilities[:, column] = expit(log_positive - log_negative)
    return probabilities


def probabilities_to_reason_labels(probabilities, threshold=REASON_THRESHOLD):
    """Threshold labels independently; choose one best reason if a row is all zero."""
    probabilities = np.asarray(probabilities)
    predictions = (probabilities >= threshold).astype(int)
    fallback_rows = ~predictions.any(axis=1)
    if fallback_rows.any():
        positions = np.flatnonzero(fallback_rows)
        predictions[positions, probabilities[positions].argmax(axis=1)] = 1
    return predictions, fallback_rows


def evaluate_reason_predictions(y_true, probabilities):
    predictions, fallback_rows = probabilities_to_reason_labels(probabilities)
    per_reason = f1_score(y_true, predictions, average=None, zero_division=0)
    metrics = pd.Series(
        [
            f1_score(y_true, predictions, average="samples", zero_division=0),
            f1_score(y_true, predictions, average="macro", zero_division=0),
            *per_reason,
        ],
        index=["Sample-wise F1", "Macro F1", *REASONS],
        name="Validation",
    )
    return metrics, predictions, int(fallback_rows.sum())

## Fit the generative classifiers and evaluate validation

This trains ten density models at most: positive and negative for each of the five reasons. Validation examples are scored directly from the learned likelihoods and training priors. The nonempty fallback is applied consistently so Kaggle predictions cannot be blank.

In [18]:
class_conditional_models = fit_class_conditional_gmms(
    X_labeled, y_labeled, REASONS,
    max_components_per_class=MAX_COMPONENTS_PER_CLASS,
    covariance_type=COVARIANCE_TYPE, n_init=N_INIT, max_iter=MAX_ITER,
    reg_covar=REG_COVAR, random_state=RANDOM_STATE,
)
validation_probabilities = predict_reason_probabilities(class_conditional_models, X_val, REASONS)
y_val = part1_val[REASONS].to_numpy(dtype=int)
validation_metrics, validation_predictions, validation_fallback_count = evaluate_reason_predictions(
    y_val, validation_probabilities
)
display(validation_metrics.round(4).to_frame())
print("Validation rows using highest-probability fallback:", validation_fallback_count)

,Validation
Sample-wise F1,0.5983
Macro F1,0.3435
Safety,0.4615
Funding,0.0000
Efficacy,0.3704
Recruitment,0.7477
Other,0.1379


Validation rows using highest-probability fallback: 4


## Inspect class support and mixture convergence

Rare reasons may have few positive training examples. The table shows the positive and negative counts, the actual component count used for each, and whether each fitted class density converged.

In [19]:
diagnostics = []
for reason in REASONS:
    for value, class_name in [(1, "positive"), (0, "negative")]:
        fitted_class = class_conditional_models[reason]["class_models"][value]
        diagnostics.append({
            "Reason": reason,
            "Class": class_name,
            "Training examples": fitted_class["training_count"],
            "GMM components": fitted_class["n_components"],
            "Converged": fitted_class["converged"],
            "EM iterations": fitted_class["iterations"],
        })
display(pd.DataFrame(diagnostics).set_index(["Reason", "Class"]))

Training examples  GMM components  Converged  \
Reason      Class                                                    
Safety      positive                 16               2       True   
            negative                141               2       True   
Funding     positive                  8               2       True   
            negative                149               2       True   
Efficacy    positive                 17               2       True   
            negative                140               2       True   
Recruitment positive                101               2       True   
            negative                 56               2       True   
Other       positive                 25               2       True   
            negative                132               2       True   

                      EM iterations  
Reason      Class                    
Safety      positive              2  
            negative              2  
Funding     positive              2  
            negative              9  
Efficacy    positive              2  
            negative             11  
Recruitment positive              6  
            negative              2  
Other       positive              2  
            negative              8

## Prepare a Kaggle submission after reviewing validation

Choose the fitted classifier above, generate its posterior probabilities for test rows, and use the same threshold and nonempty fallback. This writes a distinct `ID,Predicted` file; it does not upload it.

The final code cell is a commented example. Running all cells will not write a submission unless you explicitly enable that call.

In [20]:
def export_class_conditional_predictions(fitted_models, X, test_df, output_path):
    if len(X) != len(test_df):
        raise ValueError("Test embeddings and trials must have matching rows and order.")
    probabilities = predict_reason_probabilities(fitted_models, X, REASONS)
    predictions, fallback_rows = probabilities_to_reason_labels(probabilities)
    submission = test_df[["ID"]].copy()
    submission["Predicted"] = [
        " ".join(reason for reason, value in zip(REASONS, row) if value)
        for row in predictions
    ]
    if submission["Predicted"].isna().any() or submission["Predicted"].eq("").any():
        raise ValueError("Every Kaggle prediction must contain at least one reason.")
    submission.to_csv(output_path, index=False)
    return submission, int(fallback_rows.sum())

In [21]:
# Optional export after reviewing validation; uncomment only when ready.
# submission_path = ATTEMPT_DIR / "part1_class_conditional_gmm_submission.csv"
# class_gmm_submission, test_fallback_count = export_class_conditional_predictions(
#     class_conditional_models, X_test, part1_test, submission_path
# )
# print("Saved:", submission_path)
# print("Test rows using highest-probability fallback:", test_fallback_count)
# class_gmm_submission.head()